In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [ ]:
# Load the dataset
df = pd.read_csv('/kaggle/input/credit-card-transactions-dataset/credit_card_transactions.csv')
print(df.head())


In [ ]:
df = df.drop(columns=['Unnamed: 0'])
print(df.head())

In [ ]:
# Display summary statistics
print(df.describe())

# Check for missing values
print(df.isnull().sum())

In [ ]:
df.info()

In [ ]:
# Drop unnecessary columns
df = df.drop(columns=['trans_date_trans_time', 'trans_num', 'unix_time', 'dob', 'first', 'last'])

# Handle missing values
df['merch_zipcode'] = df['merch_zipcode'].fillna(-1)  # Using -1 as a placeholder for missing values



In [ ]:
print(df)

In [ ]:
# Separate features and target variable
X = df.drop(columns=['is_fraud'])
y = df['is_fraud']

# List of categorical columns
categorical_features = ['merchant', 'category', 'gender', 'city', 'state', 'job']

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(), ['amt', 'cc_num', 'zip', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 'merch_zipcode'])  # Adjust numerical columns as needed
    ]
)


In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Define the pipeline with preprocessing and Logistic Regression model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Fit the model
pipeline.fit(X_train, y_train)

# Predict on the test set
y_pred = pipeline.predict(X_test)

# Evaluate the model
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('Classification Report:\n', classification_report(y_test, y_pred))

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Define categorical and numerical columns
categorical_cols = X_train.select_dtypes(include=['object']).columns
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

# Preprocessor for numerical data
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocessor for categorical data
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessor into a single ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Apply preprocessing
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


In [ ]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_processed, y_train)

# Train the model
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_resampled, y_resampled)

# Predict on the test set
y_pred = model.predict(X_test_processed)

# Evaluate the model
from sklearn.metrics import classification_report, confusion_matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, f1_score

# Predict probabilities for the test set
y_prob = model.predict_proba(X_test_processed)[:, 1]

# Compute precision, recall, and thresholds
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

# Calculate F1 scores for different thresholds
f1_scores = 2 * (precision * recall) / (precision + recall)
f1_scores[np.isnan(f1_scores)] = 0  # Handle division by zero

# Find the threshold that gives the maximum F1 score
best_threshold_index = np.argmax(f1_scores)
best_threshold = thresholds[best_threshold_index]

# Print the best threshold and corresponding F1 score
print(f'Best Threshold: {best_threshold:.4f}')
print(f'Best F1 Score: {f1_scores[best_threshold_index]:.4f}')

# Plot Precision-Recall curve and F1 score
plt.figure(figsize=(10, 6))
plt.plot(recall, precision, label='Precision-Recall Curve')
plt.axvline(x=recall[best_threshold_index], color='r', linestyle='--', label='Best Threshold')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.grid(True)
plt.show()



Precision for Class 1 is very low (0.05), indicating that when the model predicts fraud, it is rarely correct.
Recall for Class 1 is high (0.81), meaning the model identifies most of the actual fraud cases.
Overall Accuracy is 0.90, but this includes the performance on both classes. The low precision for Class 1 affects the balanced evaluation

